# Track 1, Step 2: VCF Pre-Filtering

This notebook implements Step 2 of the Variant Prediction pipeline. The goal is to strip out sequencing artifacts and massive amounts of noise from the raw ~5 million variant VCF (`data/WGS_EX2312012_HGWCNDSX7.vcf.gz`).

### Process Overview:
1. **Filter 1 (Quality Control & Allelic Balance):** We enforce strict thresholds based on the machine's confidence and biological likelihood:
   - `FILTER`: Must be `PASS` or missing.
   - `GQ` (Genotype Quality): >= 20 (99% confidence).
   - `DP` (Depth of Coverage): >= 10.
   - `AB` (Allelic Balance): For heterozygous calls, the alternate allele must represent 30% to 70% of the total reads to rule out contamination or mapping artifacts.

2. **Filter 2 (Common Variant Subtraction):** We perform an *allele-aware* intersection against the `dbSNP Common` dataset. Since MVA is ultra-rare, we drop variants present in dbSNP Common.

3. **Output Generation:** We directly output a filtered VCF file (`data/track1_step2_rare_variants.vcf.gz`). This perfectly aligns with the required downstream format for Step 3 Annotation. We also inject the Allelic Balance metric into the VCF `INFO` column as `AB_FILTER`.

### Download dbSNP Common Reference Dataset
The "Common Variant Subtraction" step relies on the **dbSNP Common (GRCh38)** dataset. This curated ~1.5 GB file acts as a fast, lightweight filter because it ONLY contains variants known to have a Minor Allele Frequency (MAF) >= 1% in at least one major population.

Run the cell below to download it to your `data/` folder if you haven't already.

In [1]:
%%bash
# Navigate to the data directory
cd ../data

# Download the dbSNP Common VCF if it doesn't exist
if [ ! -f "common_all_20180418.vcf.gz" ]; then
    echo "Downloading dbSNP Common VCF (approx 1.5GB)..."
    wget ftp://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh38p7/VCF/common_all_20180418.vcf.gz
else
    echo "common_all_20180418.vcf.gz already exists."
fi

# Download the tabix index if it doesn't exist
if [ ! -f "common_all_20180418.vcf.gz.tbi" ]; then
    echo "Downloading dbSNP Common VCF Index..."
    wget ftp://ftp.ncbi.nih.gov/snp/organisms/human_9606_b151_GRCh38p7/VCF/common_all_20180418.vcf.gz.tbi
else
    echo "common_all_20180418.vcf.gz.tbi already exists."
fi


common_all_20180418.vcf.gz already exists.
common_all_20180418.vcf.gz.tbi already exists.


In [2]:
import os
import cyvcf2 # cyvcf2 is used because it's a fast C-based wrapper for VCF parsing, which is essential when iterating over 5 million rows.
import time # time is used to track how long the filtering process takes.

# Define and verify data paths. 
# Why: Ensuring the files exist before starting prevents the script from failing halfway through.
input_vcf = "../data/WGS_EX2312012_HGWCNDSX7.vcf.gz"
dbsnp_vcf = "../data/common_all_20180418.vcf.gz"
output_rare_vcf = "../data/track1_step2_rare_variants.vcf.gz"

assert os.path.exists(input_vcf), f"File not found: {input_vcf}"
assert os.path.exists(dbsnp_vcf), f"File not found: {dbsnp_vcf}"
print(f"Data files found.")


Data files found.


## Filtering (Removing "Typos" and Common Variants)
We process the VCF line by line using `cyvcf2` to apply both intrinsic QC and extrinsic filtering in a single pass to save time and disk I/O.

### How is `AB_FILTER` calculated?
**Allelic Balance (AB)** is mathematically found by dividing the number of reads supporting the alternate allele (`FORMAT_AD_ALT`) by the total number of reads spanning that specific position (`FORMAT_AD_REF` + `FORMAT_AD_ALT`).
*   **Why we need it:** In a true biological heterozygous mutation, you would expect roughly 50% of the sequencer's reads to show the reference sequence and 50% to show the mutation. If the Allelic Balance is highly skewed, it strongly implies a sequencer mapping artifact or sample contamination. We strictly require AB to be between 0.30 and 0.70 for heterozygous variants.

### How is Filter 2 performed?
For variants surviving the intrinsic checks, we query `dbSNP Common` at the exact coordinate. If a variant matches POS, REF, and at least one ALT exactly, it is considered common and is dropped.

In [4]:
# Open the input VCF for reading.
vcf_in = cyvcf2.VCF(input_vcf)
vcf_dbsnp = cyvcf2.VCF(dbsnp_vcf)

# Modify the header to include our custom AB_FILTER
vcf_in.add_info_to_header({'ID': 'AB_FILTER', 'Description': 'Allelic Balance for Heterozygous calls', 'Type': 'Float', 'Number': '1'})

# Create VCF writer
vcf_out = cyvcf2.Writer(output_rare_vcf, vcf_in)

start_time = time.time()

# Trackers for filtering statistics
stats = {
    'total_variants': 0,
    'dropped_filter': 0,
    'dropped_missing_metrics': 0,
    'dropped_gq': 0,
    'dropped_dp': 0,
    'dropped_gt': 0,
    'dropped_ab': 0,
    'passed_intrinsic': 0,
    'dropped_common': 0,
    'passed_rare': 0
}

# Iterate through every variant in the VCF file.
for variant in vcf_in:
    stats['total_variants'] += 1
    
    # 1. Filter out variants that failed the sequencer's internal quality checks.
    if variant.FILTER is not None and variant.FILTER != "PASS":
        stats['dropped_filter'] += 1
        continue
        
    # Extract formats array for our sample.
    gq_array = variant.format('GQ')
    dp_array = variant.format('DP')
    ad_array = variant.format('AD') # [REF_depth, ALT_depth, ...]
    
    # Skip if any of these critical metrics are missing.
    if gq_array is None or dp_array is None or ad_array is None:
        stats['dropped_missing_metrics'] += 1
        continue
        
    gq = gq_array[0][0]
    dp = dp_array[0][0]
    
    # 2. Apply hard depth and quality cutoffs.
    if gq < 20:
        stats['dropped_gq'] += 1
        continue
    if dp < 10:
        stats['dropped_dp'] += 1
        continue
        
    # 3. Check Genotype (GT).
    gt_type = variant.gt_types[0]
    if gt_type == 0 or gt_type == 2: # Ignore HOM_REF and UNKNOWN
        stats['dropped_gt'] += 1
        continue
        
    # 4. Calculate Allelic Balance (AB).
    ref_depth = ad_array[0][0]
    alt_depth = ad_array[0][1] if len(ad_array[0]) > 1 else 0
    calc_dp = ref_depth + alt_depth
    
    if calc_dp == 0:
        stats['dropped_ab'] += 1
        continue
        
    ab = alt_depth / calc_dp
    
    # 5. Filter based on Allelic Balance.
    if gt_type == 1: # Heterozygous (expected 50% REF, 50% ALT)
        if ab < 0.30 or ab > 0.70:
            stats['dropped_ab'] += 1
            continue
    elif gt_type == 3: # Homozygous ALT (expected ~100% ALT)
        if ab < 0.90:
            stats['dropped_ab'] += 1
            continue
            
    stats['passed_intrinsic'] += 1

    # 6. Filter 2: Common Variant Subtraction
    is_common = False
    region = f"{variant.CHROM}:{variant.POS}-{variant.POS}"
    for dbsnp_var in vcf_dbsnp(region):
        if dbsnp_var.POS == variant.POS and dbsnp_var.REF == variant.REF:
            if set(variant.ALT).intersection(set(dbsnp_var.ALT)):
                is_common = True
                break

    if is_common:
        stats['dropped_common'] += 1
        continue

    # 7. If surviving, annotate with AB and write to VCF
    variant.INFO["AB_FILTER"] = round(ab, 3)
    vcf_out.write_record(variant)
    stats['passed_rare'] += 1

vcf_in.close()
vcf_dbsnp.close()
vcf_out.close()

end_time = time.time()
print(f"Processed {stats['total_variants']:,} total variants in {end_time - start_time:.2f} seconds.")
print(f"\n--- FILTERING STATISTICS ---")
print(f"Dropped by 'PASS' flag:      {stats['dropped_filter']:,}")
print(f"Dropped missing metrics:     {stats['dropped_missing_metrics']:,}")
print(f"Dropped by GQ < 20:          {stats['dropped_gq']:,}")
print(f"Dropped by DP < 10:          {stats['dropped_dp']:,}")
print(f"Dropped by GT (Ref/Unk):     {stats['dropped_gt']:,}")
print(f"Dropped by Allelic Balance:  {stats['dropped_ab']:,}")
print(f"----------------------------")
print(f"Variants surviving Intrinsic: {stats['passed_intrinsic']:,}")
print(f"Dropped as Common (dbSNP):   {stats['dropped_common']:,}")
print(f"Final Rare Variants:         {stats['passed_rare']:,} ({(stats['passed_rare']/stats['total_variants'])*100:.1f}%)")
print(f"Filtered VCF saved directly to: {output_rare_vcf}")


/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:263-263
  for dbsnp_var in vcf_dbsnp(region):
/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:310-310
  for dbsnp_var in vcf_dbsnp(region):
/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:750-750
  for dbsnp_var in vcf_dbsnp(region):
/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:1438-1438
  for dbsnp_var in vcf_dbsnp(region):
/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:13830-13830
  for dbsnp_var in vcf_dbsnp(region):
/tmp/ipykernel_20676/3715698507.py:89: UserWarning: no intervals found for b'../data/common_all_20180418.vcf.gz' at M:14470-14470
  for dbsnp_var in vcf_dbsnp(region):
/tmp/i

Processed 5,012,204 total variants in 405.64 seconds.

--- FILTERING STATISTICS ---
Dropped by 'PASS' flag:      271,414
Dropped missing metrics:     0
Dropped by GQ < 20:          29,727
Dropped by DP < 10:          34,646
Dropped by GT (Ref/Unk):     0
Dropped by Allelic Balance:  248,144
----------------------------
Variants surviving Intrinsic: 4,428,273
Dropped as Common (dbSNP):   3,836,934
Final Rare Variants:         591,339 (11.8%)
Filtered VCF saved directly to: ../data/track1_step2_rare_variants.vcf.gz


## Reviewing the Final Output
Let's use `cyvcf2` again to read the first few variants of our output VCF to verify the new `AB_FILTER` is present in the INFO field and the rare variants.

In [5]:
vcf_check = cyvcf2.VCF(output_rare_vcf)
print(f"Successfully loaded preview of {output_rare_vcf}\n")

count = 0
for variant in vcf_check:
    print(f"CHROM: {variant.CHROM}, POS: {variant.POS}, REF: {variant.REF}, ALT: {variant.ALT}")
    print(f"INFO: {variant.INFO.get('AB_FILTER')} (AB_FILTER), DP: {variant.format('DP')[0][0]}, GQ: {variant.format('GQ')[0][0]}")
    print("-" * 40)
    count += 1
    if count >= 5:
        break

vcf_check.close()


Successfully loaded preview of ../data/track1_step2_rare_variants.vcf.gz

CHROM: 1, POS: 16298, REF: C, ALT: ['T']
INFO: 0.41499999165534973 (AB_FILTER), DP: 41, GQ: 99
----------------------------------------
CHROM: 1, POS: 98921, REF: AG, ALT: ['A']
INFO: 0.6919999718666077 (AB_FILTER), DP: 26, GQ: 99
----------------------------------------
CHROM: 1, POS: 98999, REF: T, ALT: ['TTTTA']
INFO: 1.0 (AB_FILTER), DP: 21, GQ: 63
----------------------------------------
CHROM: 1, POS: 99166, REF: C, ALT: ['T']
INFO: 0.3190000057220459 (AB_FILTER), DP: 47, GQ: 99
----------------------------------------
CHROM: 1, POS: 99180, REF: T, ALT: ['C']
INFO: 0.32600000500679016 (AB_FILTER), DP: 46, GQ: 99
----------------------------------------
